# Notebook C v11 Hardened — Adapter Validation and Submission Packaging

Purpose: consume Notebook B LoRA adapter output, enforce provenance, validate rank, and create a minimal `submission.zip`.

Hardening after v11/v12:
- tracks the extracted source zip path,
- deletes stale local `submission.zip` and extraction dirs before discovery,
- ignores stale `/kaggle/working/submission.zip` as an adapter source,
- hard-fails unless the selected adapter directory or extracted zip contains `EXPECTED_ADAPTER_TAG`.

In [ ]:
import json
import math
import os
import re
import shutil
import sys
import time
import zipfile
from pathlib import Path

import pandas as pd

RUN_VLLM_VALIDATION = False
VALIDATION_N = 3

# Change this for every new Notebook B experiment.
EXPECTED_ADAPTER_TAG = "adapter_sft64_v1_1"

WORKING_DIR = Path('/kaggle/working')
ADAPTER_EXTRACT_DIR = WORKING_DIR / 'adapter_for_validation_v11_hardened'
SUBMISSION_ZIP = WORKING_DIR / 'submission.zip'
DEBUG_CSV = WORKING_DIR / 'notebook_c_debug_predictions.csv'
MANUAL_ADAPTER_DIR = None
EXTRACTED_ZIP_PATH = None

print('RUN_VLLM_VALIDATION:', RUN_VLLM_VALIDATION)
print('EXPECTED_ADAPTER_TAG:', EXPECTED_ADAPTER_TAG)
print('WORKING_DIR:', WORKING_DIR)
print('/kaggle/input exists:', Path('/kaggle/input').exists())

In [ ]:
# Clean stale local artifacts before adapter discovery.
for p in [SUBMISSION_ZIP, ADAPTER_EXTRACT_DIR]:
    if p.exists():
        if p.is_dir():
            shutil.rmtree(p)
        else:
            p.unlink()
        print('Deleted stale artifact:', p)

In [ ]:
print('Top-level /kaggle/input entries:')
if Path('/kaggle/input').exists():
    for p in sorted(Path('/kaggle/input').iterdir()):
        print(' -', p)
else:
    print(' - /kaggle/input does not exist')

print('\nAdapter-related files visible under /kaggle/input and /kaggle/working:')
visible = []
for root in [Path('/kaggle/input'), WORKING_DIR]:
    if not root.exists():
        continue
    for pattern in [
        '**/adapter_config.json',
        '**/adapter_model.safetensors',
        '**/adapter_model.bin',
        f'**/{EXPECTED_ADAPTER_TAG}.zip',
        '**/adapter_sft64_v1_1.zip',
        '**/adapter_smoke_v1_1.zip',
    ]:
        visible.extend(root.glob(pattern))

for p in sorted(set(visible))[:200]:
    print(' -', p)
print('Count:', len(set(visible)))

In [ ]:
def _is_adapter_dir(d: Path) -> bool:
    return (d / 'adapter_config.json').exists() and (
        (d / 'adapter_model.safetensors').exists() or (d / 'adapter_model.bin').exists()
    )

def discover_adapter_dir():
    global EXTRACTED_ZIP_PATH
    candidate_dirs = []
    zip_candidates = []

    if MANUAL_ADAPTER_DIR is not None:
        candidate_dirs.append(Path(MANUAL_ADAPTER_DIR))

    search_roots = [Path('/kaggle/input/notebooks'), Path('/kaggle/input'), WORKING_DIR]
    for root in search_roots:
        if not root.exists():
            continue
        for d in root.glob(f'**/{EXPECTED_ADAPTER_TAG}'):
            if d.is_dir():
                candidate_dirs.append(d)
        for cfg in root.glob('**/adapter_config.json'):
            if EXPECTED_ADAPTER_TAG in str(cfg.parent):
                candidate_dirs.append(cfg.parent)
        zip_candidates.extend(root.glob(f'**/{EXPECTED_ADAPTER_TAG}.zip'))

    print('Zip candidates:')
    for z in sorted(set(zip_candidates))[:50]:
        print(' -', z)

    unique_zips = sorted(set(zip_candidates), key=lambda p: (0 if EXPECTED_ADAPTER_TAG in str(p) else 1, str(p)))
    for zpath in unique_zips:
        try:
            if ADAPTER_EXTRACT_DIR.exists():
                shutil.rmtree(ADAPTER_EXTRACT_DIR)
            ADAPTER_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zpath, 'r') as zf:
                zf.extractall(ADAPTER_EXTRACT_DIR)
            EXTRACTED_ZIP_PATH = zpath
            for cfg in ADAPTER_EXTRACT_DIR.glob('**/adapter_config.json'):
                candidate_dirs.insert(0, cfg.parent)
            print('Extracted adapter zip candidate:', zpath)
            break
        except Exception as e:
            print('Could not extract zip candidate:', zpath, repr(e))

    seen = set()
    unique_dirs = []
    for d in candidate_dirs:
        d = Path(d)
        key = str(d)
        if key not in seen:
            seen.add(key)
            unique_dirs.append(d)

    print('\nAdapter directory candidates:')
    for d in unique_dirs[:80]:
        print(' -', d, 'exists=', d.exists(), 'is_adapter=', _is_adapter_dir(d) if d.exists() else False)

    for d in unique_dirs:
        if _is_adapter_dir(d):
            return d
    return None

ADAPTER_DIR = discover_adapter_dir()
if ADAPTER_DIR is None:
    raise FileNotFoundError(f'Could not find adapter files for expected tag {EXPECTED_ADAPTER_TAG}.')

selected_path = str(ADAPTER_DIR)
source_zip_path = str(EXTRACTED_ZIP_PATH or '')
if EXPECTED_ADAPTER_TAG not in selected_path and EXPECTED_ADAPTER_TAG not in source_zip_path:
    raise RuntimeError(
        f'Wrong adapter selected. Expected {EXPECTED_ADAPTER_TAG}. '
        f'ADAPTER_DIR={selected_path}, EXTRACTED_ZIP_PATH={source_zip_path}'
    )

print('\nSelected ADAPTER_DIR:', ADAPTER_DIR)
print('EXTRACTED_ZIP_PATH:', EXTRACTED_ZIP_PATH)
for p in sorted(ADAPTER_DIR.iterdir()):
    if p.is_file():
        print(' -', p.name, f'{p.stat().st_size/1024/1024:.2f} MB')

In [ ]:
cfg_path = ADAPTER_DIR / 'adapter_config.json'
with open(cfg_path, 'r', encoding='utf-8') as f:
    adapter_cfg = json.load(f)

print(json.dumps(adapter_cfg, indent=2)[:2000])
rank = adapter_cfg.get('r', adapter_cfg.get('rank', None))
if rank is not None:
    print('Detected LoRA rank:', rank)
    assert int(rank) <= 32, f'LoRA rank {rank} exceeds competition max rank 32'
else:
    print('Warning: could not find rank field `r`; inspect manually.')

assert _is_adapter_dir(ADAPTER_DIR)
print('Adapter structure validation passed.')

In [ ]:
if SUBMISSION_ZIP.exists():
    SUBMISSION_ZIP.unlink()

required_or_allowed = ['adapter_config.json', 'adapter_model.safetensors', 'adapter_model.bin']
with zipfile.ZipFile(SUBMISSION_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for name in required_or_allowed:
        p = ADAPTER_DIR / name
        if p.exists():
            zf.write(p, arcname=name)

with zipfile.ZipFile(SUBMISSION_ZIP, 'r') as zf:
    names = zf.namelist()

print('Created:', SUBMISSION_ZIP, f'{SUBMISSION_ZIP.stat().st_size/1024/1024:.2f} MB')
print('Zip contents:', names)
assert 'adapter_config.json' in names
assert ('adapter_model.safetensors' in names) or ('adapter_model.bin' in names)
assert all('/' not in n.strip('/') for n in names), 'Adapter files should be at zip root, not nested.'
print('Minimal submission.zip packaging validation passed.')

## Future runs

For Notebook A v2 + Notebook B v2, change `EXPECTED_ADAPTER_TAG` to the exact Notebook B output name, for example:

```python
EXPECTED_ADAPTER_TAG = 'adapter_sft_v2_bit_512'
```